# Financial Engineering with Python: LBO Modeling from Entry to Exit

This notebook rebuilds the logic of a leveraged buyout model in Python using the same classroom case as the Excel model discussed in class.

The point is not to learn Python syntax. The point is to understand how an LBO works economically and to use Python as a transparent analytical engine.

**Roadmap**
- Inputs and assumptions
- Entry valuation and purchase price
- Sources & Uses
- Operating projection
- Cash flow available for debt repayment
- Debt schedule and deleveraging
- Credit monitoring
- Exit valuation and sponsor returns
- Sensitivity analysis
- Streamlit productization demo
- Optional AI-generated reporting layer
        


## B. Deal context and modeling objective

A leveraged buyout asks a simple question with important consequences:

> If we acquire this company with debt and equity, and the business evolves according to our underwriting assumptions, what return does the sponsor earn?

That question can be broken into a few practical concepts:

- **LBO**: an acquisition funded with a mix of debt and sponsor equity.
- **EBITDA**: a measure of operating performance before financing and non-cash depreciation.
- **Enterprise Value**: the value of the business operations, independent of capital structure.
- **Sources & Uses**: the schedule that explains where the acquisition money comes from and where it goes.
- **Deleveraging**: the reduction of debt over time through cash generation.
- **IRR**: the annualized return earned by the sponsor.
- **MOIC**: the ratio of equity value realized to equity invested.

The logic of the notebook follows the same sequence an analyst would use in practice: price the business, fund the deal, project the operating engine, sweep cash into debt repayment, monitor credit quality, and finally measure the sponsor return.
        


### Code setup

The notebook uses a small shared LBO engine so that the notebook and the Streamlit demo stay fully aligned. The modeling logic remains visible here: the helper module only keeps the calculations reusable, while the notebook remains the teaching asset.
        


In [ ]:
# Standard imports for the notebook environment and charting.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

try:
    from IPython.display import Markdown, display
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

# This makes the notebook work whether it is launched from the repo root or from the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.gemini_reporting import generate_investment_commentary
from utils.lbo_engine import run_lbo_model

# Keep tables readable for classroom discussion.
pd.options.display.float_format = "{:,.2f}".format
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {
    "navy": "#14324A",
    "teal": "#1F6F78",
    "sand": "#D7C3A3",
    "slate": "#5A6C7D",
    "green": "#2D7D46",
    "red": "#A6473C",
}


def eur_millions(value, _position=None):
    # Most outputs in the model are in EUR millions, so the same formatter is reused across charts.
    return f"EUR {value:,.0f}m"


def style_axes(axis):
    # A cleaner frame makes the finance story easier to read from the chart.
    axis.spines[["top", "right"]].set_visible(False)


def add_bar_labels(axis, bars, offset=10, fmt="{:.1f}"):
    # Bar labels help students reconcile the visual with the exact values.
    for bar in bars:
        height = bar.get_height()
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset,
            fmt.format(height),
            ha="center",
            va="bottom",
            fontsize=9,
            color=COLORS["navy"],
        )


def format_inputs_table(table):
    # Convert raw numbers into the units students expect when reading a deal sheet.
    formatted = table.copy()
    formatted["Display Value"] = formatted.apply(
        lambda row: (
            f"{row['Value'] * 100:.1f}%" if row["Unit"] == "%"
            else f"{row['Value']:.1f}x" if row["Unit"] == "x"
            else f"EUR {row['Value']:.2f}" if row["Unit"] == "EUR/share"
            else f"{row['Value']:.1f}m" if row["Unit"] == "m shares"
            else f"EUR {row['Value']:,.1f}m" if row["Unit"] == "EURm"
            else f"{int(row['Value'])}" if row["Unit"] == "year"
            else f"{row['Value']}"
        ),
        axis=1,
    )
    return formatted[["Group", "Assumption", "Display Value"]]


def format_metric_table(table):
    # Some metrics are percentages, some are multiples, and some are money values.
    formatted = table.copy()
    formatted["Display Value"] = formatted.apply(
        lambda row: (
            f"{row['Value'] * 100:.1f}%" if row["Metric"] in {"Takeover premium", "IRR"}
            else f"{row['Value']:.2f}x" if row["Metric"] == "MOIC"
            else f"EUR {row['Value']:,.2f}" if row["Metric"] == "Implied offer price per share"
            else f"EUR {row['Value']:,.1f}m"
        ),
        axis=1,
    )
    return formatted[["Metric", "Display Value"]]


def plot_heatmap(table, title, ax, percent_values=False):
    # Heatmaps are useful in class because they reveal the full return pattern at a glance.
    values = table.to_numpy(dtype=float)
    image = ax.imshow(values, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(np.arange(len(table.columns)))
    ax.set_xticklabels(table.columns)
    ax.set_yticks(np.arange(len(table.index)))
    ax.set_yticklabels(table.index)
    ax.set_title(title)
    ax.set_xlabel(table.columns.name)
    ax.set_ylabel(table.index.name)

    midpoint = values.mean()
    for row_idx, row_value in enumerate(table.index):
        for col_idx, col_value in enumerate(table.columns):
            display_value = table.loc[row_value, col_value]
            label = f"{display_value * 100:.1f}%" if percent_values else f"{display_value:.2f}x"
            ax.text(
                col_idx,
                row_idx,
                label,
                ha="center",
                va="center",
                fontsize=9,
                color="white" if display_value > midpoint else COLORS["navy"],
            )

    plt.colorbar(image, ax=ax, shrink=0.82)


# Run the base-case LBO once and keep every intermediate table available for the walkthrough.
results = run_lbo_model()
inputs_table = results["inputs_table"]
entry_valuation = results["entry_valuation"]
sources_and_uses = results["sources_and_uses"]
operating_projection = results["operating_projection"]
debt_schedule = results["debt_schedule"]
credit_metrics = results["credit_metrics"]
returns_summary = results["returns_summary"]
value_creation_bridge = results["value_creation_bridge"]
sensitivities = results["sensitivities"]

returns_lookup = returns_summary.set_index("Metric")["Value"]
entry_lookup = entry_valuation.set_index("Metric")["Value"]
        


## C. Inputs and assumptions

Before calculating anything, we need to be clear about what is fixed in the case and what is assumed by the analyst.

**What do we calculate here?** We organize the full transaction perimeter: deal data, valuation assumptions, financing mix, operating starting point, taxes, fees, and exit assumptions.

**Why does it matter?** LBO models are highly assumption-driven. Good modeling starts by making the assumptions explicit rather than hiding them inside formulas.

**How should we interpret this section?** This is the underwriting sheet of the transaction. If an output looks too optimistic or too weak later on, this is the first place to revisit.

For alignment with the classroom Excel file, the historical data and the first three projected operating years are matched to the class case. Sensitivity analysis then perturbs that same base case rather than introducing a different forecasting architecture.
        


In [ ]:
# Show the base-case assumptions in the same grouped logic used in class.
display(format_inputs_table(inputs_table))
        


## D. Entry valuation and purchase price

Once the assumptions are clear, the first analytical question is pricing: what are we paying for the business, and what does that imply for the shareholders?

**What do we calculate?** Historical EBITDA, Enterprise Value, implied equity value, offer price per share, and takeover premium.

**Why does it matter?** The entry price is one of the strongest drivers of eventual returns. A good business can still produce a weak LBO outcome if the sponsor pays too much.

**How should we interpret it?** In this case, the 7.0x entry multiple produces a meaningful premium to the unaffected share price, which is realistic for a control transaction.
        


In [ ]:
# Start with the valuation outputs before moving to the chart.
display(format_metric_table(entry_valuation))

# The bridge keeps the logic explicit: Enterprise Value minus net debt equals equity value.
bridge = pd.DataFrame(
    {
        "Label": ["Enterprise Value", "Net Debt", "Equity Value"],
        "Value": [
            entry_lookup["Entry Enterprise Value"],
            entry_lookup["Less: Net debt"],
            entry_lookup["Implied Equity Value"],
        ],
        "Color": [COLORS["navy"], COLORS["red"], COLORS["teal"]],
    }
)

fig, ax = plt.subplots(figsize=(8.2, 4.4))
# Plot the bridge as three bars so students can see the transition from firm value to shareholder value.
bars = ax.bar(bridge["Label"], bridge["Value"], color=bridge["Color"], width=0.62)
for idx, row in bridge.iterrows():
    y = row["Value"] + 35 if row["Value"] >= 0 else row["Value"] - 70
    ax.text(idx, y, f"{row['Value']:,.1f}", ha="center", color=COLORS["navy"], fontsize=10)
ax.set_title("Entry valuation bridge", pad=12)
ax.set_ylabel("EUR millions")
ax.yaxis.set_major_formatter(FuncFormatter(eur_millions))
ax.axhline(0, color="#666666", linewidth=0.8)
ax.text(
    1.95,
    entry_lookup["Implied Equity Value"] * 0.88,
    f"Offer price: EUR {entry_lookup['Implied offer price per share']:.2f}\nPremium: {entry_lookup['Takeover premium'] * 100:.1f}%",
    ha="right",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#cccccc"},
)
style_axes(ax)
plt.show()
        


## E. Sources & Uses

After valuing the target, the next question is funding: how is the acquisition actually paid for?

**What do we calculate?** The uses of funds, the sources of funds, and the sponsor equity check needed to close the deal.

**Why does it matter?** This schedule translates valuation into transaction mechanics. It is where the model stops being a valuation exercise and becomes a financing exercise.

**How should we interpret it?** The class case deliberately keeps the structure simple: senior debt, subordinated debt, and a residual sponsor equity contribution.
        


In [ ]:
# Show the exact transaction funding schedule first.
display(sources_and_uses.fillna(""))

# Split the table into uses and sources so both sides can be compared visually.
uses = sources_and_uses["Uses"].dropna().drop("Total Uses")
sources = sources_and_uses["Sources"].dropna().drop("Total Sources")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
axes[0].barh(uses.index, uses.values, color=[COLORS["sand"], COLORS["slate"], COLORS["red"]])
axes[0].set_title("Uses of funds")
axes[1].barh(sources.index, sources.values, color=[COLORS["navy"], COLORS["teal"], COLORS["green"]])
axes[1].set_title("Sources of funds")

for axis in axes:
    axis.set_xlabel("EUR millions")
    axis.xaxis.set_major_formatter(FuncFormatter(eur_millions))
    style_axes(axis)

axes[1].text(
    0.98,
    0.08,
    f"Sponsor equity = {sources['Sponsor equity'] / sources.sum():.0%} of total sources",
    transform=axes[1].transAxes,
    ha="right",
    color=COLORS["navy"],
)
plt.tight_layout()
plt.show()
        


## F. Operating projection

With the deal funded, the model moves into the operating engine of the business.

**What do we calculate?** EBIT, D&A, EBITDA, taxes, capital expenditure, change in operating working capital, and cash flow available for debt repayment.

**Why does it matter?** An LBO is not repaid with accounting profit alone. It is repaid with cash left after taxes and reinvestment needs.

**How should we interpret it?** The business is growing EBITDA across the projection period, but not every euro of EBITDA becomes debt-repaying cash. That gap is one of the key lessons of the LBO model.
        


In [ ]:
# Extend the operating table with interest, taxes, and debt-paying cash so the full operating engine is visible.
projection_view = operating_projection.copy()
projection_view["Interest expense"] = debt_schedule["Interest expense"]
projection_view["Tax expense"] = debt_schedule["Tax expense"]
projection_view["Cash flow available for debt repayment"] = debt_schedule["Cash flow available for debt repayment"]
display(projection_view)

# The charts focus on projected years because that is where the LBO economics play out.
projected = operating_projection.drop(index="Historical")
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.4))

axes[0].plot(projected.index, projected["EBITDA"], marker="o", linewidth=2.6, color=COLORS["navy"], label="EBITDA")
axes[0].bar(projected.index, projected["EBIT"], color=COLORS["teal"], alpha=0.32, label="EBIT")
axes[0].set_title("Operating engine: EBIT and EBITDA")
axes[0].set_ylabel("EUR millions")
axes[0].yaxis.set_major_formatter(FuncFormatter(eur_millions))
axes[0].legend(frameon=False)
axes[0].annotate(
    f"Historical EBITDA: {operating_projection.loc['Historical', 'EBITDA']:.1f}",
    xy=(0, projected["EBITDA"].iloc[0]),
    xytext=(-0.35, projected["EBITDA"].iloc[0] + 12),
    textcoords="data",
    arrowprops={"arrowstyle": "->", "color": COLORS["navy"]},
    color=COLORS["navy"],
)
axes[0].annotate(
    f"Year 3 EBITDA: {projected['EBITDA'].iloc[-1]:.1f}",
    xy=(2, projected["EBITDA"].iloc[-1]),
    xytext=(1.45, projected["EBITDA"].iloc[-1] + 14),
    textcoords="data",
    arrowprops={"arrowstyle": "->", "color": COLORS["navy"]},
    color=COLORS["navy"],
)
style_axes(axes[0])

# This is the cash available for the sweep after interest, taxes, working capital, and capex.
cash_bars = axes[1].bar(debt_schedule.index, debt_schedule["Cash flow available for debt repayment"], color=COLORS["green"], width=0.58)
axes[1].plot(debt_schedule.index, debt_schedule["Cash flow available for debt repayment"], color=COLORS["navy"], marker="o", linewidth=1.5)
axes[1].set_title("Cash available for debt repayment")
axes[1].set_ylabel("EUR millions")
axes[1].yaxis.set_major_formatter(FuncFormatter(eur_millions))
add_bar_labels(axes[1], cash_bars, offset=1.5)
style_axes(axes[1])

plt.tight_layout()
plt.show()
        


In [ ]:
# Use the exit year to show one full bridge from accounting profit to debt-paying cash.
bridge_year = "Year 3"
# Positive items add to cash and negative items consume cash.
components = pd.Series(
    {
        "EBIT": operating_projection.loc[bridge_year, "EBIT"],
        "D&A": operating_projection.loc[bridge_year, "D&A"],
        "Interest": -debt_schedule.loc[bridge_year, "Interest expense"],
        "Taxes": -debt_schedule.loc[bridge_year, "Tax expense"],
        "Change in OWC": operating_projection.loc[bridge_year, "Change in OWC"],
        "Capex": -operating_projection.loc[bridge_year, "Capex"],
    }
)

fig, ax = plt.subplots(figsize=(9.2, 4.2))
ax.bar(
    components.index,
    components.values,
    color=[COLORS["teal"], COLORS["sand"], COLORS["red"], COLORS["red"], COLORS["slate"], COLORS["red"]],
)
ax.axhline(debt_schedule.loc[bridge_year, "Cash flow available for debt repayment"], color=COLORS["green"], linestyle="--", linewidth=2)
ax.text(
    len(components) - 0.15,
    debt_schedule.loc[bridge_year, "Cash flow available for debt repayment"] + 2,
    f"Debt repayment capacity = EUR {debt_schedule.loc[bridge_year, 'Cash flow available for debt repayment']:.1f}m",
    ha="right",
    color=COLORS["green"],
)
ax.set_title("Year 3 bridge from operating profit to debt-paying cash")
ax.set_ylabel("EUR millions")
ax.yaxis.set_major_formatter(FuncFormatter(eur_millions))
style_axes(ax)
plt.xticks(rotation=18)
plt.show()
        


## G. Debt schedule and deleveraging

Once we know how much cash the business generates, we can trace how that cash moves through the capital structure.

**What do we calculate?** Opening debt balances, interest expense, debt repayment, ending balances, and any residual cash after the sweep.

**Why does it matter?** This is the core LBO transmission mechanism. Operating cash flow improves the credit profile because it is swept into debt reduction.

**How should we interpret it?** The class case uses a clean rule: repay senior debt first, then subordinated debt only after senior debt has been exhausted.
        


In [ ]:
# Keep the debt schedule explicit so students can trace beginning balance, interest, repayment, and ending balance.
debt_view = debt_schedule[[
    "Beginning senior debt",
    "Interest expense on senior debt",
    "Repayment of senior debt",
    "Ending senior debt",
    "Beginning subordinated debt",
    "Interest expense on subordinated debt",
    "Repayment of subordinated debt",
    "Ending subordinated debt",
    "Ending cash balance",
]]
display(debt_view)

# Total debt is the stacked sum of senior and subordinated balances.
total_debt = debt_schedule["Ending senior debt"] + debt_schedule["Ending subordinated debt"]

fig, ax = plt.subplots(figsize=(9.2, 4.4))
ax.bar(debt_schedule.index, debt_schedule["Ending senior debt"], color=COLORS["navy"], label="Senior debt")
ax.bar(
    debt_schedule.index,
    debt_schedule["Ending subordinated debt"],
    bottom=debt_schedule["Ending senior debt"],
    color=COLORS["sand"],
    label="Subordinated debt",
)
ax.plot(debt_schedule.index, total_debt, color=COLORS["red"], linewidth=2, marker="o", label="Total debt")
ax.set_title("Debt evolution and deleveraging path")
ax.set_ylabel("EUR millions")
ax.yaxis.set_major_formatter(FuncFormatter(eur_millions))
ax.legend(frameon=False, ncol=3, loc="upper right")
ax.annotate(
    f"Debt reduced by {total_debt.iloc[0] - total_debt.iloc[-1]:.1f}m by Year 3",
    xy=(2, total_debt.iloc[-1]),
    xytext=(1.1, total_debt.iloc[-1] + 95),
    textcoords="data",
    arrowprops={"arrowstyle": "->", "color": COLORS["red"]},
    color=COLORS["red"],
)
style_axes(ax)
plt.show()
        


## H. Credit monitoring and debt capacity logic

The debt schedule tells us how much leverage remains, but lenders look at leverage relative to earnings and debt service burden.

**What do we calculate?** Total Debt / EBITDA, EBITDA / Interest Expense, percent of total debt repaid, and percent of senior debt repaid.

**Why does it matter?** These metrics indicate whether the capital structure is becoming safer over time.

**How should we interpret it?** In a healthy LBO, leverage falls while interest coverage rises. That is exactly the pattern we want to see before exit.
        


In [ ]:
# Format the debt repayment ratios as percentages before showing the lender-facing metrics.
credit_view = credit_metrics.copy()
credit_view["% of total debt repaid"] = credit_view["% of total debt repaid"].map(lambda x: f"{x * 100:.1f}%")
credit_view["% of senior debt repaid"] = credit_view["% of senior debt repaid"].map(lambda x: f"{x * 100:.1f}%")
display(credit_view)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.3))
axes[0].plot(credit_metrics.index, credit_metrics["Total Debt / EBITDA"], marker="o", linewidth=2.5, color=COLORS["red"])
axes[0].set_title("Leverage trend")
axes[0].set_ylabel("x")
axes[0].annotate(
    f"Year 3: {credit_metrics['Total Debt / EBITDA'].iloc[-1]:.2f}x",
    xy=(2, credit_metrics["Total Debt / EBITDA"].iloc[-1]),
    xytext=(1.35, credit_metrics["Total Debt / EBITDA"].iloc[-1] + 0.23),
    textcoords="data",
    arrowprops={"arrowstyle": "->", "color": COLORS["red"]},
    color=COLORS["red"],
)
style_axes(axes[0])

axes[1].plot(credit_metrics.index, credit_metrics["EBITDA / Interest expense"], marker="o", linewidth=2.5, color=COLORS["green"])
axes[1].set_title("Interest coverage trend")
axes[1].set_ylabel("x")
axes[1].annotate(
    f"Year 3: {credit_metrics['EBITDA / Interest expense'].iloc[-1]:.2f}x",
    xy=(2, credit_metrics["EBITDA / Interest expense"].iloc[-1]),
    xytext=(1.28, credit_metrics["EBITDA / Interest expense"].iloc[-1] + 0.18),
    textcoords="data",
    arrowprops={"arrowstyle": "->", "color": COLORS["green"]},
    color=COLORS["green"],
)
style_axes(axes[1])

plt.tight_layout()
plt.show()
        


## I. Exit valuation and sponsor returns

At this point, all the moving pieces come together: the operating profile, the remaining debt balance, and the exit valuation assumption.

**What do we calculate?** Exit Enterprise Value, exit net debt, exit equity value, sponsor cash flows, IRR, and MOIC.

**Why does it matter?** These are the core outputs that determine whether the deal works for the sponsor.

**How should we interpret it?** In the classroom base case, returns are created without any multiple expansion. That means the economics are driven mainly by operating improvement and deleveraging.
        


In [ ]:
# Present the headline return outputs before decomposing them visually.
display(format_metric_table(returns_summary))
display(value_creation_bridge)

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.4))
# The left chart shows the sponsor outcome, while the right chart explains what created that outcome.
left_bars = axes[0].bar(
    ["Equity invested", "Equity realized"],
    [returns_lookup["Sponsor equity invested"], returns_lookup["Exit Equity Value"]],
    color=[COLORS["navy"], COLORS["green"]],
    width=0.58,
)
axes[0].set_title("Sponsor equity invested vs realized")
axes[0].set_ylabel("EUR millions")
axes[0].yaxis.set_major_formatter(FuncFormatter(eur_millions))
add_bar_labels(axes[0], left_bars, offset=18)
axes[0].text(
    0.98,
    0.12,
    f"IRR = {returns_lookup['IRR'] * 100:.1f}%\nMOIC = {returns_lookup['MOIC']:.2f}x",
    transform=axes[0].transAxes,
    ha="right",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#cccccc"},
)
style_axes(axes[0])

axes[1].bar(
    value_creation_bridge["Driver"],
    value_creation_bridge["Value"],
    color=[COLORS["navy"], COLORS["teal"], COLORS["sand"], COLORS["green"], COLORS["red"], COLORS["navy"]],
)
axes[1].set_title("Value creation bridge")
axes[1].set_ylabel("EUR millions")
axes[1].yaxis.set_major_formatter(FuncFormatter(eur_millions))
style_axes(axes[1])
plt.setp(axes[1].get_xticklabels(), rotation=18, ha="right")

plt.tight_layout()
plt.show()
        


## J. Sensitivity analysis

A single base case is useful for learning the mechanics, but it is not enough for decision-making.

**What do we calculate?** Sensitivity tables for IRR across entry and exit multiples, and for returns across exit multiples and operating growth changes.

**Why does it matter?** This is how analysts test whether the deal is robust or fragile.

**How should we interpret it?** The sensitivity maps make it easy to see where the return profile is supported by the underlying economics and where it becomes too dependent on favorable valuation assumptions.
        


In [ ]:
# Sensitivity analysis reinforces that one base case is never enough in transaction work.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.3))
plot_heatmap(sensitivities["entry_exit_irr"], "IRR: entry multiple vs exit multiple", axes[0], percent_values=True)
plot_heatmap(sensitivities["exit_growth_irr"], "IRR: exit multiple vs EBIT growth shift", axes[1], percent_values=True)
plt.tight_layout()
plt.show()

display((sensitivities["entry_exit_irr"] * 100).round(1))
display((sensitivities["exit_growth_moic"]).round(2))
        


## K. From analysis to product: Streamlit demo

Once the analytical engine is stable, the same model can be packaged into a lightweight decision-support tool.

**What do we calculate?** Nothing new. The app reuses the same logic already built in the notebook.

**Why does it matter?** It shows students how an analyst workflow can become an interactive product without changing the underlying finance.

**How should we interpret it?** The app is not the lesson itself. It is the end-of-class productization demo that turns the same case into an interactive interface.
        


In [ ]:
# Point students to the productization layer without duplicating the app inside the notebook.
streamlit_app_path = PROJECT_ROOT / "app" / "streamlit_lbo_demo.py"
print(f"Streamlit app: {streamlit_app_path}")
print("Run locally with: streamlit run app/streamlit_lbo_demo.py")
        


## L. AI-generated investment commentary

A completed model still needs a short narrative output. That is where the optional Gemini layer comes in.

**What do we calculate?** We do not calculate new economics. We convert selected model outputs into a concise investment note.

**Why does it matter?** Analysts often need to communicate what the model says, not just show the spreadsheet or notebook.

**How should we interpret it?** The AI is a reporting assistant, not the model itself. If no API key is configured, the notebook simply skips this step without breaking.
        


In [ ]:
# The AI layer reads the completed model outputs; it does not create the valuation itself.
commentary = generate_investment_commentary(results)

if commentary["success"]:
    display(Markdown(commentary["message"]))
else:
    print(commentary["message"])
        


## M. Key takeaways from the LBO build

Three ideas drive the result of this case:

- **Operating performance** matters because EBITDA growth expands the earnings base on which the exit valuation is applied.
- **Deleveraging** matters because cash flow repayment transfers value from lenders to the sponsor over time.
- **Exit valuation** matters because even a strong operating case can be undermined by an unfavorable exit multiple.

The broader lesson is that an LBO is not just a capital structure exercise. It is a disciplined way of linking operating assumptions, financing mechanics, and sponsor returns into one coherent analytical framework.
        
